# GRPO fine-tuning for NanoVLM in MiniGrid EmptyEnv

This notebook runs two GRPO experiments: direct action generation and text + action generation.


## 1. Install dependencies
Install the Python packages required for MiniGrid, NanoVLM, and training.


In [ ]:
!pip install -q gymnasium minigrid accelerate sentencepiece safetensors gcsfs tqdm
!pip install -q transformers==4.45.0 datasets==3.5.0


## 2. Clone NanoVLM
Clone the NanoVLM repository and switch to its directory.


In [ ]:
import os
import subprocess

repo_path = "/content/nanoVLM"

if not os.path.isdir(repo_path):
    subprocess.run(
        ["git", "clone", "https://github.com/huggingface/nanoVLM.git", repo_path],
        check=True,
    )

%cd /content/nanoVLM


## 3. Patch tokenizer initialization
Patch `data/processors.py` so that additional VLM tokens are registered correctly.


In [ ]:
from pathlib import Path

processors_path = Path("data/processors.py")

old_get_tokenizer = '''def get_tokenizer(name, extra_special_tokens=None, chat_template=None):
    if name not in TOKENIZERS_CACHE:
        tokenizer_init_kwargs = {"use_fast": True}
        if extra_special_tokens is not None:
            tokenizer_init_kwargs["extra_special_tokens"] = extra_special_tokens
        if chat_template is not None:
            tokenizer_init_kwargs["chat_template"] = chat_template
        tokenizer = AutoTokenizer.from_pretrained(name, **tokenizer_init_kwargs,)
        tokenizer.pad_token = tokenizer.eos_token
        TOKENIZERS_CACHE[name] = tokenizer
    return TOKENIZERS_CACHE[name]
'''

new_get_tokenizer = '''def get_tokenizer(name, extra_special_tokens=None, chat_template=None):
    if name not in TOKENIZERS_CACHE:
        tokenizer = AutoTokenizer.from_pretrained(name, use_fast=True)

        if extra_special_tokens is not None:
            new_tokens = list(extra_special_tokens.values())
            tokenizer.add_tokens(new_tokens, special_tokens=True)

            for token_name, token_str in extra_special_tokens.items():
                setattr(tokenizer, token_name, token_str)
                token_id = tokenizer.convert_tokens_to_ids(token_str)
                setattr(tokenizer, token_name + "_id", token_id)

        if chat_template is not None:
            tokenizer.chat_template = chat_template

        tokenizer.pad_token = tokenizer.eos_token
        TOKENIZERS_CACHE[name] = tokenizer

    return TOKENIZERS_CACHE[name]
'''

source = processors_path.read_text()

if old_get_tokenizer not in source:
    raise RuntimeError(
        "Original get_tokenizer implementation was not found. "
        "The repository file may have changed."
    )

source = source.replace(old_get_tokenizer, new_get_tokenizer)

processors_path.write_text(source)

print("Patched data/processors.py:get_tokenizer")


## 4. Configure CUDA memory allocation
Set a CUDA allocator option that helps reduce memory fragmentation in Colab.


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


## 5. Load SFT checkpoints from Google Drive
Mount Google Drive and copy the SFT checkpoints used to initialize GRPO. Adjust the paths if needed.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Update these paths if your checkpoints are stored elsewhere.
!cp -r "/content/drive/MyDrive/checkpoints/step_100" "/content/step_100"
!cp -r "/content/drive/MyDrive/checkpoints/step_400_text" "/content/step_400"


## 6. GRPO with direct action generation
The model is initialized from the action-only SFT checkpoint and directly generates one action id: `0`, `1`, or `2`.


In [ ]:
import os
import re
import gc
import csv
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from minigrid.wrappers import RGBImgObsWrapper

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor, get_image_string


# =====================================================
# CONFIG
# =====================================================

SFT_CHECKPOINT = "/content/step_100"
OUTPUT_DIR = "/content/grpo_generate_action"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TILE_SIZE = 42

PROMPT = """Look at the image.

Action IDs:
0 = left
1 = right
2 = forward

Choose the best action to reach the goal.

Answer with only one number."""

NUM_UPDATES = 15
GROUP_SIZE = 2
MAX_EPISODE_STEPS = 30

LR = 1e-7

EVAL_EVERY = 1
EVAL_EPISODES = 10
EVAL_SEEDS = list(range(EVAL_EPISODES))

STEP_PENALTY = -0.01
SUCCESS_REWARD = 1.0

ACTION_TO_TEXT = {
    0: "0",
    1: "1",
    2: "2",
}

ACTION_LABELS = ["0", "1", "2"]


# =====================================================
# LOAD MODEL
# =====================================================

gc.collect()

model = VisionLanguageModel.from_pretrained(SFT_CHECKPOINT).to(DEVICE)
model.train()

tokenizer = get_tokenizer(
    model.cfg.lm_tokenizer,
    model.cfg.vlm_extra_tokens,
    model.cfg.lm_chat_template
)

resize_to_max_side_len = False
if hasattr(model.cfg, "resize_to_max_side_len"):
    resize_to_max_side_len = model.cfg.resize_to_max_side_len

image_processor = get_image_processor(
    model.cfg.max_img_size,
    model.cfg.vit_img_size,
    resize_to_max_side_len
)

# Train only lightweight projection or adapter modules when available.
for name, p in model.named_parameters():
    p.requires_grad = False

trainable_keywords = ["mp", "projector", "adapter", "connector"]

num_trainable = 0

for name, p in model.named_parameters():
    if any(k in name.lower() for k in trainable_keywords):
        p.requires_grad = True
        num_trainable += p.numel()

if num_trainable == 0:
    print("WARNING: no MP/projector params found, training all parameters on CPU will be very slow.")
    for p in model.parameters():
        p.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    weight_decay=0.0
)

print("Device:", DEVICE)
print("Trainable params:", sum(p.numel() for p in trainable_params))


# =====================================================
# ENV
# =====================================================

def make_env():
    env = gym.make(
        "MiniGrid-Empty-Random-6x6-v0",
        render_mode="rgb_array"
    )
    env = RGBImgObsWrapper(env, tile_size=TILE_SIZE)
    return env


# =====================================================
# INPUTS
# =====================================================

def prepare_prompt_inputs(obs_image):
    img = Image.fromarray(obs_image).convert("RGB")

    processed_image, ratio = image_processor(img)

    if (
        not hasattr(tokenizer, "global_image_token")
        and ratio[0] * ratio[1] == len(processed_image) - 1
    ):
        processed_image = processed_image[1:]

    image_string = get_image_string(
        tokenizer,
        [ratio],
        model.cfg.mp_image_token_length
    )

    messages = [
        {
            "role": "user",
            "content": image_string + PROMPT
        }
    ]

    input_ids = tokenizer.apply_chat_template(
        [messages],
        tokenize=True,
        add_generation_prompt=True
    )

    input_ids = torch.tensor(input_ids, device=DEVICE)

    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)

    images = processed_image.to(DEVICE)

    return input_ids, images


# =====================================================
# GENERATE ACTION
# =====================================================

def parse_action(text):
    text = text.strip()
    match = re.search(r"[012]", text)

    if match is None:
        return 2, "fallback"

    action = int(match.group(0))

    if action in [0, 1, 2]:
        return action, str(action)

    return 2, "fallback"


def generate_action(obs_image):
    input_ids, images = prepare_prompt_inputs(obs_image)

    model.eval()

    with torch.inference_mode():
        gen = model.generate(
            input_ids,
            images,
            max_new_tokens=3
        )

    output_text = tokenizer.batch_decode(
        gen,
        skip_special_tokens=True
    )[0]

    action, parsed = parse_action(output_text)

    model.train()

    return action, parsed, output_text


# =====================================================
# LOGPROB OF CHOSEN ACTION
# =====================================================

def forward_logits(input_ids, images):
    attention_mask = torch.ones_like(input_ids, device=DEVICE)

    out = model(
        input_ids,
        images,
        attention_mask=attention_mask
    )

    if isinstance(out, torch.Tensor):
        logits = out
    elif hasattr(out, "logits"):
        logits = out.logits
    elif isinstance(out, dict) and "logits" in out:
        logits = out["logits"]
    elif isinstance(out, (tuple, list)):
        logits = out[0]
    else:
        raise ValueError(f"Cannot extract logits from output type: {type(out)}")

    return logits


def action_logprob(obs_image, action):
    input_ids, images = prepare_prompt_inputs(obs_image)

    logits = forward_logits(input_ids, images)

    last_logits = logits[:, -1, :]

    action_text = ACTION_TO_TEXT[action]

    action_token_ids = tokenizer.encode(
        action_text,
        add_special_tokens=False
    )

    action_token_id = action_token_ids[0]

    log_probs = torch.log_softmax(last_logits, dim=-1)

    logprob = log_probs[0, action_token_id]

    return logprob


# =====================================================
# ROLLOUT USING model.generate()
# =====================================================

def rollout_episode(seed=None):
    env = make_env()
    obs, _ = env.reset(seed=seed)

    observations = []
    actions = []

    total_return = 0.0
    steps = 0
    success = False

    terminated = False
    truncated = False

    while not terminated and not truncated and steps < MAX_EPISODE_STEPS:
        obs_image = obs["image"].copy()

        action, parsed, raw = generate_action(obs_image)

        observations.append(obs_image)
        actions.append(action)

        obs, reward, terminated, truncated, _ = env.step(action)

        if terminated and reward > 0:
            shaped_reward = SUCCESS_REWARD
            success = True
        else:
            shaped_reward = STEP_PENALTY

        total_return += shaped_reward
        steps += 1

    env.close()

    return {
        "observations": observations,
        "actions": actions,
        "return": total_return,
        "length": steps,
        "success": success,
    }


# =====================================================
# EVALUATION USING model.generate()
# =====================================================

def evaluate_policy():
    env = make_env()

    successes = 0
    returns = []
    lengths = []

    model.eval()

    for seed in EVAL_SEEDS:
        obs, _ = env.reset(seed=seed)

        total_return = 0.0
        steps = 0
        success = False

        terminated = False
        truncated = False

        while not terminated and not truncated and steps < MAX_EPISODE_STEPS:
            action, parsed, raw = generate_action(obs["image"])

            obs, reward, terminated, truncated, _ = env.step(action)

            if terminated and reward > 0:
                shaped_reward = SUCCESS_REWARD
                success = True
            else:
                shaped_reward = STEP_PENALTY

            total_return += shaped_reward
            steps += 1

        if success:
            successes += 1

        returns.append(total_return)
        lengths.append(steps)

    env.close()
    model.train()

    return {
        "success_rate": successes / len(EVAL_SEEDS),
        "mean_return": float(np.mean(returns)),
        "mean_length": float(np.mean(lengths)),
    }


# =====================================================
# BASELINE
# =====================================================

print("Evaluating SFT baseline with model.generate()...")
baseline = evaluate_policy()
print("SFT baseline:", baseline)


# =====================================================
# GRPO LOOP
# =====================================================

logs = []

for update in tqdm(range(1, NUM_UPDATES + 1), desc="GRPO-generate"):

    base_seed = 10_000 + update

    trajectories = []

    for _ in range(GROUP_SIZE):
        traj = rollout_episode(seed=base_seed)
        trajectories.append(traj)

    returns = torch.tensor(
        [t["return"] for t in trajectories],
        dtype=torch.float32,
        device=DEVICE
    )

    if returns.std(unbiased=False) > 1e-6:
        advantages = (
            returns - returns.mean()
        ) / (returns.std(unbiased=False) + 1e-8)
    else:
        advantages = returns - returns.mean()

    losses = []

    for traj, adv in zip(trajectories, advantages):
        logprobs = []

        for obs_image, action in zip(
            traj["observations"],
            traj["actions"]
        ):
            lp = action_logprob(
                obs_image,
                action
            )
            logprobs.append(lp)

        if len(logprobs) == 0:
            continue

        traj_logprob = torch.stack(logprobs).sum()

        loss = -adv.detach() * traj_logprob
        losses.append(loss)

    if len(losses) == 0:
        continue

    loss = torch.stack(losses).mean()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()

    row = {
        "update": update,
        "loss": float(loss.item()),
        "train_group_return": float(returns.mean().item()),
        "train_group_success": float(np.mean([t["success"] for t in trajectories])),
        "train_group_length": float(np.mean([t["length"] for t in trajectories])),
    }

    if update % EVAL_EVERY == 0:
        eval_metrics = evaluate_policy()

        row.update({
            "eval_success_rate": eval_metrics["success_rate"],
            "eval_return": eval_metrics["mean_return"],
            "eval_length": eval_metrics["mean_length"],
        })

        print(
            f"Update {update}: "
            f"loss={row['loss']:.4f}, "
            f"train_return={row['train_group_return']:.3f}, "
            f"train_success={row['train_group_success']:.3f}, "
            f"eval_SR={eval_metrics['success_rate']:.3f}, "
            f"eval_return={eval_metrics['mean_return']:.3f}"
        )

    logs.append(row)

    gc.collect()


# =====================================================
# SAVE + PLOTS
# =====================================================

csv_path = "/content/grpo_generate_logs.csv"

fieldnames = sorted(set(k for row in logs for k in row.keys()))

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(logs)

print("Saved logs:", csv_path)


eval_rows = [
    r for r in logs
    if "eval_success_rate" in r
]

if len(eval_rows) > 0:
    xs = [0] + [r["update"] for r in eval_rows]
    srs = [baseline["success_rate"]] + [r["eval_success_rate"] for r in eval_rows]
    rets = [baseline["mean_return"]] + [r["eval_return"] for r in eval_rows]

    plt.figure(figsize=(8, 5))
    plt.plot(xs, srs, marker="o")
    plt.xlabel("GRPO update")
    plt.ylabel("Success rate")
    plt.title("GRPO-action using model.generate(): Success Rate")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(xs, rets, marker="o")
    plt.xlabel("GRPO update")
    plt.ylabel("Average return")
    plt.title("GRPO-action using model.generate(): Average Return")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


## 7. GRPO with text + action generation
The model is initialized from the text+action SFT checkpoint and generates `State`, `Plan`, and `Action` fields.


In [ ]:
import os
import re
import gc
import csv
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from minigrid.wrappers import RGBImgObsWrapper

from models.vision_language_model import VisionLanguageModel
from data.processors import get_tokenizer, get_image_processor, get_image_string


# =====================================================
# CONFIG
# =====================================================

SFT_CHECKPOINT = "/content/step_400"
OUTPUT_DIR = "/content/grpo_text_action"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TILE_SIZE = 42

PROMPT = """Look at the image.

Describe the current situation in 1-2 short sentences.
Then choose the best action to reach the goal.

Action IDs:
0 = left
1 = right
2 = forward

Use exactly this format:
State: ...
Plan: ...
Action: 0/1/2"""

NUM_UPDATES = 15
GROUP_SIZE = 2
MAX_EPISODE_STEPS = 30

LR = 1e-7

EVAL_EVERY = 1
EVAL_EPISODES = 10
EVAL_SEEDS = list(range(EVAL_EPISODES))

STEP_PENALTY = -0.01
SUCCESS_REWARD = 1.0

ACTION_TO_TEXT = {
    0: "0",
    1: "1",
    2: "2",
}


# =====================================================
# LOAD MODEL
# =====================================================

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = VisionLanguageModel.from_pretrained(SFT_CHECKPOINT).to(DEVICE)
model.train()

tokenizer = get_tokenizer(
    model.cfg.lm_tokenizer,
    model.cfg.vlm_extra_tokens,
    model.cfg.lm_chat_template
)

resize_to_max_side_len = False
if hasattr(model.cfg, "resize_to_max_side_len"):
    resize_to_max_side_len = model.cfg.resize_to_max_side_len

image_processor = get_image_processor(
    model.cfg.max_img_size,
    model.cfg.vit_img_size,
    resize_to_max_side_len
)

for name, p in model.named_parameters():
    p.requires_grad = False

trainable_keywords = ["mp", "projector", "adapter", "connector"]

num_trainable = 0

for name, p in model.named_parameters():
    if any(k in name.lower() for k in trainable_keywords):
        p.requires_grad = True
        num_trainable += p.numel()

if num_trainable == 0:
    print("WARNING: no MP/projector params found, training all parameters.")
    for p in model.parameters():
        p.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    weight_decay=0.0
)

print("Device:", DEVICE)
print("Trainable params:", sum(p.numel() for p in trainable_params))


# =====================================================
# ENV
# =====================================================

def make_env():
    env = gym.make(
        "MiniGrid-Empty-Random-6x6-v0",
        render_mode="rgb_array"
    )
    env = RGBImgObsWrapper(env, tile_size=TILE_SIZE)
    return env


# =====================================================
# INPUTS
# =====================================================

def prepare_prompt_inputs(obs_image):
    img = Image.fromarray(obs_image).convert("RGB")

    processed_image, ratio = image_processor(img)

    if (
        not hasattr(tokenizer, "global_image_token")
        and ratio[0] * ratio[1] == len(processed_image) - 1
    ):
        processed_image = processed_image[1:]

    image_string = get_image_string(
        tokenizer,
        [ratio],
        model.cfg.mp_image_token_length
    )

    messages = [
        {
            "role": "user",
            "content": image_string + PROMPT
        }
    ]

    input_ids = tokenizer.apply_chat_template(
        [messages],
        tokenize=True,
        add_generation_prompt=True
    )

    input_ids = torch.tensor(input_ids, device=DEVICE)

    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)

    images = processed_image.to(DEVICE)

    return input_ids, images


# =====================================================
# PARSE TEXT+ACTION OUTPUT
# =====================================================

def parse_action(text):
    if text is None:
        return 2, "fallback_forward"

    text = text.strip()

    match = re.search(
        r"Action\s*:\s*([012])",
        text,
        flags=re.IGNORECASE
    )

    if match is not None:
        action = int(match.group(1))
        return action, f"action_{action}"

    # Fallback: use the last standalone action id in the generated text.
    matches = re.findall(r"\b[012]\b", text)

    if len(matches) > 0:
        action = int(matches[-1])
        return action, f"fallback_digit_{action}"

    return 2, "fallback_forward"


def generate_action(obs_image):
    input_ids, images = prepare_prompt_inputs(obs_image)

    model.eval()

    with torch.inference_mode():
        gen = model.generate(
            input_ids,
            images,
            max_new_tokens=120
        )

    # Decode only newly generated tokens first.
    input_len = input_ids.shape[1]
    answer_tokens = gen[0][input_len:]

    output_text = tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ).strip()

    # Fallback: decode the full generated sequence.
    if len(output_text) == 0:
        output_text = tokenizer.batch_decode(
            gen,
            skip_special_tokens=True
        )[0].strip()

    action, parsed = parse_action(output_text)

    model.train()

    return action, parsed, output_text


# =====================================================
# LOGPROB OF CHOSEN ACTION
# =====================================================

def forward_logits(input_ids, images):
    attention_mask = torch.ones_like(input_ids, device=DEVICE)

    out = model(
        input_ids,
        images,
        attention_mask=attention_mask
    )

    if isinstance(out, torch.Tensor):
        logits = out
    elif hasattr(out, "logits"):
        logits = out.logits
    elif isinstance(out, dict) and "logits" in out:
        logits = out["logits"]
    elif isinstance(out, (tuple, list)):
        logits = out[0]
    else:
        raise ValueError(f"Cannot extract logits from output type: {type(out)}")

    return logits


def action_logprob(obs_image, action):
    """
    Lightweight GRPO approximation: use the log-probability of the final
    action token instead of the full State/Plan/Action sequence.
    """

    input_ids, images = prepare_prompt_inputs(obs_image)

    logits = forward_logits(input_ids, images)

    last_logits = logits[:, -1, :]

    action_text = ACTION_TO_TEXT[action]

    action_token_ids = tokenizer.encode(
        action_text,
        add_special_tokens=False
    )

    action_token_id = action_token_ids[0]

    log_probs = torch.log_softmax(last_logits, dim=-1)

    logprob = log_probs[0, action_token_id]

    return logprob


# =====================================================
# ROLLOUT USING model.generate()
# =====================================================

def rollout_episode(seed=None):
    env = make_env()
    obs, _ = env.reset(seed=seed)

    observations = []
    actions = []
    parsed_outputs = []

    total_return = 0.0
    steps = 0
    success = False

    terminated = False
    truncated = False

    while not terminated and not truncated and steps < MAX_EPISODE_STEPS:
        obs_image = obs["image"].copy()

        action, parsed, raw = generate_action(obs_image)

        observations.append(obs_image)
        actions.append(action)
        parsed_outputs.append(parsed)

        obs, reward, terminated, truncated, _ = env.step(action)

        if terminated and reward > 0:
            shaped_reward = SUCCESS_REWARD
            success = True
        else:
            shaped_reward = STEP_PENALTY

        total_return += shaped_reward
        steps += 1

    env.close()

    return {
        "observations": observations,
        "actions": actions,
        "parsed_outputs": parsed_outputs,
        "return": total_return,
        "length": steps,
        "success": success,
    }


# =====================================================
# EVALUATION USING model.generate()
# =====================================================

def evaluate_policy():
    env = make_env()

    successes = 0
    returns = []
    lengths = []

    invalid_outputs = 0
    fallback_outputs = 0

    model.eval()

    for ep_idx, seed in enumerate(tqdm(EVAL_SEEDS, desc="Eval episodes")):
        obs, _ = env.reset(seed=seed)

        total_return = 0.0
        steps = 0
        success = False

        terminated = False
        truncated = False

        while not terminated and not truncated and steps < MAX_EPISODE_STEPS:
            action, parsed, raw = generate_action(obs["image"])

            if parsed == "fallback_forward":
                invalid_outputs += 1

            if parsed.startswith("fallback"):
                fallback_outputs += 1

            obs, reward, terminated, truncated, _ = env.step(action)

            if terminated and reward > 0:
                shaped_reward = SUCCESS_REWARD
                success = True
            else:
                shaped_reward = STEP_PENALTY

            total_return += shaped_reward
            steps += 1

        if success:
            successes += 1

        returns.append(total_return)
        lengths.append(steps)

    env.close()
    model.train()

    return {
        "success_rate": successes / len(EVAL_SEEDS),
        "mean_return": float(np.mean(returns)),
        "mean_length": float(np.mean(lengths)),
        "invalid_outputs": invalid_outputs,
        "fallback_outputs": fallback_outputs,
    }


# =====================================================
# BASELINE
# =====================================================

print("Evaluating SFT text+action baseline with model.generate()...")
baseline = evaluate_policy()
print("SFT text+action baseline:", baseline)


# =====================================================
# GRPO LOOP
# =====================================================

logs = []

for update in tqdm(range(1, NUM_UPDATES + 1), desc="GRPO-text-action"):

    base_seed = 10_000 + update

    trajectories = []

    for group_idx in range(GROUP_SIZE):
        traj = rollout_episode(
            seed=base_seed + group_idx,
        )
        trajectories.append(traj)

    returns = torch.tensor(
        [t["return"] for t in trajectories],
        dtype=torch.float32,
        device=DEVICE
    )

    if returns.std(unbiased=False) > 1e-6:
        advantages = (
            returns - returns.mean()
        ) / (returns.std(unbiased=False) + 1e-8)
    else:
        advantages = returns - returns.mean()

    losses = []

    for traj, adv in zip(trajectories, advantages):
        logprobs = []

        for obs_image, action in zip(
            traj["observations"],
            traj["actions"]
        ):
            lp = action_logprob(
                obs_image,
                action
            )
            logprobs.append(lp)

        if len(logprobs) == 0:
            continue

        traj_logprob = torch.stack(logprobs).sum()

        loss = -adv.detach() * traj_logprob
        losses.append(loss)

    if len(losses) == 0:
        continue

    loss = torch.stack(losses).mean()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()

    row = {
        "update": update,
        "loss": float(loss.item()),
        "train_group_return": float(returns.mean().item()),
        "train_group_success": float(np.mean([t["success"] for t in trajectories])),
        "train_group_length": float(np.mean([t["length"] for t in trajectories])),
        "train_fallback_outputs": int(
            sum(
                parsed.startswith("fallback")
                for t in trajectories
                for parsed in t["parsed_outputs"]
            )
        ),
    }

    if update % EVAL_EVERY == 0:
        eval_metrics = evaluate_policy()

        row.update({
            "eval_success_rate": eval_metrics["success_rate"],
            "eval_return": eval_metrics["mean_return"],
            "eval_length": eval_metrics["mean_length"],
            "invalid_outputs": eval_metrics["invalid_outputs"],
            "fallback_outputs": eval_metrics["fallback_outputs"],
        })

        print(
            f"Update {update}: "
            f"loss={row['loss']:.4f}, "
            f"train_return={row['train_group_return']:.3f}, "
            f"train_success={row['train_group_success']:.3f}, "
            f"eval_SR={eval_metrics['success_rate']:.3f}, "
            f"eval_return={eval_metrics['mean_return']:.3f}, "
            f"fallback={eval_metrics['fallback_outputs']}"
        )

    logs.append(row)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =====================================================
# SAVE + PLOTS
# =====================================================

csv_path = "/content/grpo_text_action_logs.csv"

fieldnames = sorted(set(k for row in logs for k in row.keys()))

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(logs)

print("Saved logs:", csv_path)


eval_rows = [
    r for r in logs
    if "eval_success_rate" in r
]

if len(eval_rows) > 0:
    xs = [0] + [r["update"] for r in eval_rows]
    srs = [baseline["success_rate"]] + [r["eval_success_rate"] for r in eval_rows]
    rets = [baseline["mean_return"]] + [r["eval_return"] for r in eval_rows]

    plt.figure(figsize=(8, 5))
    plt.plot(xs, srs, marker="o")
    plt.xlabel("GRPO update")
    plt.ylabel("Success rate")
    plt.title("GRPO text+action using model.generate(): Success Rate")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(xs, rets, marker="o")
    plt.xlabel("GRPO update")
    plt.ylabel("Average return")
    plt.title("GRPO text+action using model.generate(): Average Return")
    plt.grid(True)
    plt.tight_layout()
    plt.show()
